<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW10 · Session 2 · Importing JSON, Pickle, and Compressed Data

        **Course level:** developing beginner  
        **Prior learning:** portable paths and validated CSV/Excel imports from TW10 Session 1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Different formats preserve different structures. This session reads nested JSON, a trusted local pickle, compressed CSV data, and large files in chunks.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - load JSON with the standard library and pandas
- flatten nested records with json_normalize
- explain why untrusted pickle files must never be opened
- read gzip and selected files inside ZIP archives
- process a delimited file in chunks and record import provenance


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Locate data and import libraries

The standard-library modules inspect formats without hiding their structure.


In [ ]:
from pathlib import Path
import json
import pickle
import zipfile
import pandas as pd

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())


## 1. JSON: objects, arrays, and nested records

JSON represents objects, arrays, strings, numbers, Booleans, and null values. A JSON file may be a single object, a list of records, or a deeply nested document; inspect before converting it to a table.


### Load JSON with the standard library

The file contains a top-level list of dictionaries. Nested contact details remain dictionaries.


In [ ]:
json_path = DATA_DIR / "sample_records.json"
with json_path.open("r", encoding="utf-8") as handle:
    records = json.load(handle)

print("Top-level type:", type(records))
print("Record count:", len(records))
print("First record:", records[0])


### Flatten nested JSON

json_normalize creates dotted column names for nested objects. Lists remain list-valued unless a separate record path is requested.


In [ ]:
customer_table = pd.json_normalize(records, sep="_")
print(customer_table)
print(customer_table.dtypes)


### Read record-oriented JSON directly

pandas can read many regular JSON arrangements, but explicit inspection remains important for nested structures.


In [ ]:
direct_json = pd.read_json(json_path)
print(direct_json[["customer_id", "name", "active"]])


## 2. Pickle: Python-specific serialisation

Pickle can preserve Python objects and types efficiently, but loading a malicious pickle can execute arbitrary code. Open a pickle **only when it comes from a source you trust and control**. It is not a safe data-exchange format.


        ### Load the supplied trusted pickle

        This file was created by the course generator. Compare its content with the CSV version.

<div class="warning"><strong>Take care:</strong> Never call read_pickle or pickle.load on an emailed, downloaded, or otherwise untrusted file.</div>


In [ ]:
pickle_path = DATA_DIR / "sample_table.pkl"
trusted_table = pd.read_pickle(pickle_path)

csv_table = pd.read_csv(DATA_DIR / "sample_sales.csv")
print(trusted_table.head())
print("Same shape as CSV:", trusted_table.shape == csv_table.shape)


## 3. gzip compression

Compression reduces storage and transfer size. pandas can infer gzip compression from the .gz suffix while reading the underlying CSV.


### Read a gzip-compressed CSV

The compressed and uncompressed imports should have the same rows and columns.


In [ ]:
gzip_table = pd.read_csv(DATA_DIR / "sample_sales.csv.gz")
plain_table = pd.read_csv(DATA_DIR / "sample_sales.csv")

print("Gzip shape:", gzip_table.shape)
print("Equal values:", gzip_table.equals(plain_table))


## 4. ZIP archives with several members

A ZIP may contain several files. Inspect member names and open the intended member explicitly rather than assuming the first file is correct.


### Inspect and read one ZIP member

zipfile gives control over which member is passed to pandas.


In [ ]:
zip_path = DATA_DIR / "sample_bundle.zip"
with zipfile.ZipFile(zip_path, "r") as archive:
    member_names = archive.namelist()
    print("Members:", member_names)
    with archive.open("sales_inside.csv") as csv_member:
        zipped_sales = pd.read_csv(csv_member)

print(zipped_sales.head())


## 5. Chunked reading

For a file too large to fit comfortably in memory, chunksize returns an iterator of smaller DataFrames. Aggregate each chunk and combine the partial results.


### Aggregate a CSV in chunks

The example file is small for teaching, but the pattern also applies to much larger delimited files.


In [ ]:
chunk_totals = []
for chunk_number, chunk in enumerate(
    pd.read_csv(DATA_DIR / "sample_sales.csv", chunksize=3),
    start=1,
):
    chunk_revenue = (chunk["units"] * chunk["unit_price"]).sum()
    chunk_totals.append(chunk_revenue)
    print("Chunk", chunk_number, "rows:", len(chunk))

print("Total revenue:", round(sum(chunk_totals), 2))


## 6. Record provenance

Provenance records where data came from and how it was read. A minimal import log includes path, format, encoding or sheet/member, import time, row count, and important options.


### Build a reproducible import log

The log is data itself, so it can be saved or combined across sources later.


In [ ]:
provenance = pd.DataFrame([
    {
        "source": "sample_sales.csv",
        "format": "CSV",
        "options": "default comma delimiter; UTF-8",
        "rows": len(plain_table),
        "columns": plain_table.shape[1],
    },
    {
        "source": "sample_records.json",
        "format": "JSON",
        "options": "UTF-8; json_normalize with underscore separator",
        "rows": len(customer_table),
        "columns": customer_table.shape[1],
    },
])
print(provenance)


## Guided comparison: verify equivalent CSV representations

We compare a plain CSV, gzip copy, ZIP member, and trusted pickle. Equivalent tabular values may still have different inferred dtypes, so we compare both shape and content.


### Compare four imported objects

Convert date-like differences only if needed, then document any remaining type difference.


In [ ]:
comparisons = pd.DataFrame({
    "format": ["CSV", "gzip CSV", "ZIP member", "trusted pickle"],
    "rows": [len(plain_table), len(gzip_table), len(zipped_sales), len(trusted_table)],
    "columns": [
        plain_table.shape[1],
        gzip_table.shape[1],
        zipped_sales.shape[1],
        trusted_table.shape[1],
    ],
})
print(comparisons)
print("CSV equals gzip:", plain_table.equals(gzip_table))
print("CSV equals ZIP:", plain_table.equals(zipped_sales))


## Student coding lab

Use only the supplied trusted files. Do not download or open an unknown pickle.


        ### Student task 1: Inspect nested JSON

        <div class="task"><strong>Your job:</strong> Load sample_records.json, print every top-level key in the first record, and print the second customer's city.</div>

        **Check your work**

        - The file is opened as UTF-8.
- Keys come from the loaded data.
- The nested city is reached through dictionaries.


In [ ]:
# STUDENT TASK 1
task_records = []
print("Keys:", list(task_records[0]) if task_records else [])
print("Second city:", None)


        ### Student task 2: Normalise customer records

        <div class="task"><strong>Your job:</strong> Flatten the records using underscore-separated names and select customer_id, name, contact_city, and active.</div>

        **Check your work**

        - There is one row per customer.
- The nested city becomes a normal column.
- Only four requested columns remain.


In [ ]:
# STUDENT TASK 2
flat_customers = pd.DataFrame()
selected_customers = pd.DataFrame()
print(selected_customers)


        ### Student task 3: Read compressed CSV

        <div class="task"><strong>Your job:</strong> Read the gzip file, validate that it has eight rows, and calculate total units and total revenue.</div>

        **Check your work**

        - pandas reads the .gz file directly.
- The row assertion passes.
- Revenue is calculated from units times price.


In [ ]:
# STUDENT TASK 3
compressed_sales = pd.DataFrame()
total_units = 0
total_revenue = 0.0
print(total_units, total_revenue)


        ### Student task 4: Select a ZIP member safely

        <div class="task"><strong>Your job:</strong> List archive members, assert that sales_inside.csv exists, and import only that member.</div>

        **Check your work**

        - The archive is opened with a context manager.
- The expected name is checked.
- The CSV member becomes an eight-row DataFrame.


In [ ]:
# STUDENT TASK 4
archive_path = DATA_DIR / "sample_bundle.zip"
zip_members = []
selected_member_table = pd.DataFrame()
print(zip_members)
print(selected_member_table.shape)


        ### Student task 5: Aggregate in chunks

        <div class="task"><strong>Your job:</strong> Read sample_sales.csv in chunks of two rows. Accumulate row count, unit count, and revenue, then compare the row count with a normal import.</div>

        **Check your work**

        - Every chunk contributes once.
- The accumulated row count equals eight.
- Revenue agrees with a normal full-file calculation.



**Optional extension:** Create a provenance record for every imported format used in this session.


In [ ]:
# STUDENT TASK 5
accumulated_rows = 0
accumulated_units = 0
accumulated_revenue = 0.0

# Add a chunked loop.
print(accumulated_rows, accumulated_units, accumulated_revenue)


        ## Knowledge check

        Answer these without running new code first.

        1. How can nested JSON differ from a rectangular table?
2. Why is an untrusted pickle dangerous?
3. How does gzip differ from ZIP?
4. Why inspect ZIP member names?
5. When is chunked reading useful?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW11 uses business_sales_dirty.csv to profile and clean real-world data problems.
